In [ ]:
import pandas as pd

#zip file processing
import zipfile

#directory controls
import os

import io

In [ ]:
#change to your file directory
ZIP_FOLDER = "..\\datasets\\ERC20-stablecoins-002.zip"
GFC_DATA_FOLDER="..\\datasets\\gfc data"

In [ ]:
os.makedirs("../datasets/processed/erc_20", exist_ok=True)  # creates nested directories
os.makedirs("../datasets/processed/gfc_data", exist_ok=True)

In [ ]:
def describe(f, name):
    """
    To understand more about the dataframe (columns, num of null values, shape)
    Args:
        f -> String representing : file path
        name -> String : file name
    Output:
        None
    """
    if name.startswith("event_data"):
        df = pd.read_csv(f, encoding='latin-1')
    else:
        df = pd.read_csv(f)
    print(f"\n📋 All columns in {name}")
    print(df.columns.tolist())
    print(f"\n📊 First 5 rows:")
    print(df.head(5))
    print(f"\n Dataframe shape")
    print(df.shape)
    print(f"\n Number of null values")
    print(df.isna().sum())
    return df

# Processing ERC 20 Data

In [ ]:
data_dataframes = {}
with zipfile.ZipFile(ZIP_FOLDER ) as z:
    for name in z.namelist():

        if name.endswith(".zip"):
            # if it is another zipped folder
            print(f"Opening nested zip {name}")
            nested_zip = z.read(name)

            with zipfile.ZipFile(io.BytesIO(nested_zip)) as nested_z:
                for fileName in nested_z.namelist():
                    with nested_z.open(fileName) as nested_f:
                        df = pd.read_csv(f)
                        df['date'] = pd.to_datetime(df["timestamp"], unit="s") #convert unix timestamp to date time
                        df["coins"] = fileName.split("_")[0] #get the coin name as column
                        df.to_csv(f"../datasets/processed/erc_20/processed_{fileName}", index=False)
        else:
            print(f"Opening file {name}")
            # if it is a file
            with z.open(name) as f:
                df = describe(f, name)

                if ("timestamp" in df.columns): #conditions according to column name
                    df["date"] = pd.to_datetime(df["timestamp"], unit="s") #convert unix timestamp to date time
                elif ("time_stamp" in df.columns):
                    df["date"] = pd.to_datetime(df["time_stamp"], unit="s") #convert unix timestamp to date time
                df.to_csv(f"../datasets/processed/erc_20/processed_{name}", index=False)

# Processing GFC Data

In [ ]:
for filename in os.listdir(GFC_DATA_FOLDER):
    if filename.endswith(".csv"):
        file_path = os.path.join(GFC_DATA_FOLDER, filename)
        if os.path.isfile(file_path):
            print(filename)

            df = describe(file_path, filename) #understand the file
            print(f"Processing file for {df.iloc[0,1]}")
            #formating df's format
            ticker = df.iloc[0, 1] #get ticker value
            date= df.iloc[2:, 0] # get date column
            df = df.iloc[3:,:] # get the relevant dataset (from row 2 onwards)
            df['ticker'] = ticker #assigned ticker to ticker column
            df['date'] = date #assigned date to date column
            df.to_csv(f"../datasets/processed/gfc_data/processed_{filename}", index=False)
